# Qwen 기본 목소리 20종 전용 · v1.0.2

**남성 10개 + 여성 10개를 목록에서 바로 선택합니다. 참조 녹음이나 Gemini 키는 필요 없습니다.**

- 먼저 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요.
- 공유 사이트: **1번 설치 → 2번 서버 준비 → 4번 연결**.
- 무료 코랩 안에서 직접 생성: **1번 → 2번 → 3번**. 사이트에서 모든 화자를 ‘기본 목소리 20종’으로 배정하고 **기본 목소리 20종 코랩 대본 받기**로 파일을 받으세요.
- **첫 사용 때만** 선택한 목소리를 VoiceDesign으로 만들고 저장합니다. Base가 저장한 목소리를 참조해 이후 모든 대사를 생성합니다.
- 이 20개는 한국어 목소리 설명을 바탕으로 만드는 프리셋입니다. 실제 음색과 연령감은 결과에 따라 다릅니다.
- 첫 생성은 **모델 2개의 다운로드와 선택한 목소리 준비** 때문에 추가 시간이 듭니다. GPU에는 한 모델씩만 올립니다.
- Base는 각 목소리에 정해진 기본 말투를 사용합니다. CustomVoice식 감정 지시문은 지원하지 않습니다.
- 생성한 목소리를 **6번 셀에서 보관**하세요. 새 런타임에서는 **1번 → 6번 복원 → 2번** 순서로 같은 음성을 재사용할 수 있습니다.
- Cosy, GPT와 별도의 코랩에서 사용하세요.
- 무료 코랩은 GPU 배정·시간이 제한되고 외부 웹 화면 위주의 사용이 제한됩니다. [Colab 공식 안내](https://research.google.com/colaboratory/faq.html)

[공유 사이트](https://voice-studio.streamlit.app/) · [Qwen 공식 설명](https://github.com/QwenLM/Qwen3-TTS)

코드 작성 후 실행 테스트나 샘플 음성 생성은 수행하지 않았습니다.

**v1.0.1:** 설치·준비 진행 로그를 셀에 직접 표시합니다. 4번 셀은 저장된 연결 주소를 다시 읽어 표시합니다. `returncode=0`만으로 완료를 안내하지 않습니다.

**v1.0.2:** 공유 사이트의 Qwen 메뉴를 이 20종 전용 코랩 하나로 통일했습니다. 모델·음질 설정은 변경하지 않았습니다.


In [ ]:
#@title 1️⃣ 기본 목소리 20종 설치 (처음 한 번)
from pathlib import Path
from runpy import run_path
qwen_root = Path('/content/voice_studio_qwen_bank_v1')
qwen_root.mkdir(parents=True, exist_ok=True)
qwen_sources = {
    'qwen_voicebank_colab_server.py': '"""Personal Colab runner for 20 designed Korean voices + Qwen 1.7B Base.\n\nThis module is copied into the notebook, so a downloaded notebook is complete.\nImporting this module never installs packages or starts inference.\n"""\nimport argparse\nfrom collections import deque\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport queue\nimport re\nimport secrets\nimport socket\nimport subprocess\nimport sys\nimport threading\nimport time\nimport urllib.error\nimport urllib.request\nimport wave\nfrom qwen_voicebank_catalog import VOICEBANK, BASE_MODEL, BANK_REVISION\n\nROOT = Path(\'/content/voice_studio_qwen_bank_v1\')\nPYTHON = ROOT / \'venv/bin/python\'\nSTATE = ROOT / \'state.json\'\nMODEL_ID = BASE_MODEL\nSERVICE = \'voice-studio-qwen-voicebank\'\nVERSION = \'1.0.0\'\nVOICES = tuple(VOICEBANK)\n\n\ndef save_json(path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False), encoding=\'utf-8\')\n    temporary.chmod(0o600)\n    temporary.replace(path)\n\n\ndef local_request(path, data=None):\n    state = json.loads(STATE.read_text())\n    url = f"http://127.0.0.1:{state[\'port\']}/v1/{state[\'token\']}" + path\n    request = urllib.request.Request(url, data=None if data is None else json.dumps(data).encode(),\n                                     headers={\'Content-Type\': \'application/json\'})\n    with urllib.request.urlopen(request, timeout=30) as response:\n        return json.load(response)\n\n\ndef setup():\n    ROOT.mkdir(parents=True, exist_ok=True, mode=0o700)\n    marker = ROOT / \'installed.json\'\n    if marker.is_file() and PYTHON.is_file() and json.loads(marker.read_text()).get(\'version\') == VERSION:\n        print(\'✅ Qwen 설치 파일이 준비되어 있습니다. 2번 셀을 실행하세요.\', flush=True)\n        return\n    import shutil\n    if not shutil.which(\'nvidia-smi\') or subprocess.run([\'nvidia-smi\'], capture_output=True).returncode != 0:\n        raise RuntimeError(\'런타임 → 런타임 유형 변경 → T4 GPU를 선택한 뒤 다시 실행하세요.\')\n    print(\'▶ 1/3 Qwen 전용 설치 공간 준비\', flush=True)\n    # No ensurepip/apt Python-version dependency; pip --python can bootstrap a venv.\n    subprocess.check_call([sys.executable, \'-m\', \'venv\', \'--without-pip\', str(ROOT / \'venv\')])\n    pip = [sys.executable, \'-m\', \'pip\', \'--python\', str(PYTHON)]\n    print(\'▶ 2/3 PyTorch 설치 · 다운로드 중에는 시간이 걸립니다.\', flush=True)\n    subprocess.check_call(pip + [\'install\', \'torch==2.8.0\', \'torchaudio==2.8.0\',\n                                \'--index-url\', \'https://download.pytorch.org/whl/cu126\'])\n    print(\'▶ 3/3 Qwen 기본 목소리와 연결 도구 설치\', flush=True)\n    subprocess.check_call(pip + [\'install\', \'qwen-tts==0.1.1\', \'fastapi>=0.115,<1\',\n                                \'uvicorn>=0.30,<1\', \'soundfile>=0.12,<1\'])\n    # sox is an upstream runtime dependency, separate from its Python wrapper.\n    subprocess.check_call([\'apt-get\', \'-qq\', \'update\'])\n    subprocess.check_call([\'apt-get\', \'-qq\', \'install\', \'-y\', \'sox\', \'ffmpeg\'])\n    save_json(marker, {\'version\': VERSION})\n    print(\'✅ 설치 완료. 2번 셀에서 모델을 준비하세요.\', flush=True)\n\n\ndef start():\n    if not PYTHON.is_file() or not (ROOT / \'installed.json\').is_file():\n        raise RuntimeError(\'먼저 1번 설치 셀을 끝까지 실행해주세요.\')\n    state = {}\n    if STATE.is_file():\n        state = json.loads(STATE.read_text())\n        try:\n            status = local_request(\'/health\')\n            if status.get(\'service\') == SERVICE and status.get(\'version\') == VERSION and status.get(\'ready\'):\n                print(\'✅ 기존 Qwen 모델을 재사용합니다. 3번(직접 생성) 또는 4번(사이트 연결)으로 이동하세요.\')\n                return\n        except (OSError, ValueError):\n            pass\n        # An earlier cell may have been interrupted while its child still loads.\n        pid = state.get(\'pid\')\n        command = Path(f\'/proc/{pid}/cmdline\') if isinstance(pid, int) else None\n        if command and command.is_file() and b\'qwen_voicebank_colab_server.py\\x00serve\' in command.read_bytes():\n            print(\'▶ 이미 시작한 Qwen 모델 준비를 이어서 기다립니다.\', flush=True)\n            wait_ready(pid)\n            return\n    with socket.socket() as sock:\n        sock.bind((\'127.0.0.1\', 0))\n        port = sock.getsockname()[1]\n    state = {\'port\': port, \'token\': secrets.token_urlsafe(32), \'version\': VERSION}\n    save_json(STATE, state)\n    environment = dict(os.environ, PYTHONUNBUFFERED=\'1\', TOKENIZERS_PARALLELISM=\'false\',\n                       HF_HOME=str(ROOT / \'hf_cache\'), PYTORCH_CUDA_ALLOC_CONF=\'expandable_segments:True\')\n    with (ROOT / \'server.log\').open(\'w\') as log:\n        process = subprocess.Popen([str(PYTHON), str(Path(__file__).resolve()), \'serve\'],\n                                   stdout=log, stderr=subprocess.STDOUT, env=environment, start_new_session=True)\n    state[\'pid\'] = process.pid\n    save_json(STATE, state)\n    wait_ready(process.pid)\n\n\ndef wait_ready(pid):\n    print(\'▶ 기본 목소리 20종 서버를 시작합니다. 모델과 선택한 목소리는 첫 생성 때 준비합니다.\', flush=True)\n    started, last_log = time.monotonic(), \'\'\n    while time.monotonic() - started < 1800:\n        try:\n            status = local_request(\'/health\')\n            if status.get(\'ready\') and status.get(\'service\') == SERVICE:\n                print(f"✅ 서버 준비 완료 · {status[\'gpu\']} · {status[\'precision\']} · GPU 배치 최대 {status[\'batch_size\']}개", flush=True)\n                print(\'선택한 목소리를 처음 사용할 때 VoiceDesign → Base 준비 시간이 추가됩니다. 같은 목소리는 재사용합니다.\', flush=True)\n                print(\'공유 사이트는 4번 셀, 코랩 안에서 직접 생성하려면 3번 셀을 실행하세요.\', flush=True)\n                return\n        except (OSError, ValueError):\n            pass\n        log = (ROOT / \'server.log\').read_text(errors=\'replace\')[-3500:]\n        if log != last_log:\n            print(log if not last_log else log[-1200:], flush=True)\n            last_log = log\n        status_path = Path(f\'/proc/{pid}/status\')\n        if not status_path.exists() or re.search(r\'^State:\\s+Z\', status_path.read_text(), re.M):\n            raise RuntimeError(\'Qwen 모델 준비가 중단되었습니다. 바로 위 오류를 확인해주세요.\')\n        time.sleep(3)\n    raise RuntimeError(\'모델 준비가 30분을 넘었습니다. server.log의 다운로드·GPU 오류를 확인해주세요. 자동으로 재시작하지 않았습니다.\')\n\n\ndef tunnel():\n    status = local_request(\'/health\')\n    if not status.get(\'ready\') or status.get(\'service\') != SERVICE:\n        raise RuntimeError(\'먼저 2번 셀의 모델 준비 완료를 기다려주세요.\')\n    state = json.loads(STATE.read_text())\n    previous = state.get(\'tunnel_pid\')\n    command = Path(f\'/proc/{previous}/cmdline\') if isinstance(previous, int) else None\n    if command and command.is_file() and b\'cloudflared\' in command.read_bytes() and state.get(\'public_url\'):\n        print(\'✅ 프로그램 연결 주소 (아래 전체를 Qwen 코랩 주소 칸에 넣으세요)\\n\' + state[\'public_url\'])\n        return\n    binary = ROOT / \'cloudflared\'\n    if not binary.is_file():\n        print(\'▶ Cloudflare 연결 도구 다운로드\', flush=True)\n        urllib.request.urlretrieve(\'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\', binary)\n        binary.chmod(0o700)\n    log_path = ROOT / \'tunnel.log\'\n    with log_path.open(\'w\') as log:\n        process = subprocess.Popen([str(binary), \'tunnel\', \'--url\', f"http://127.0.0.1:{state[\'port\']}",\n                                    \'--no-autoupdate\'], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n    state[\'tunnel_pid\'] = process.pid\n    save_json(STATE, state)\n    for _ in range(60):\n        output = log_path.read_text(errors=\'replace\')\n        match = re.search(r\'https://[a-z0-9-]+\\.trycloudflare\\.com\', output)\n        if match:\n            state[\'public_url\'] = match.group() + \'/v1/\' + state[\'token\']\n            save_json(STATE, state)\n            print(\'✅ Qwen 공유 사이트 연결 준비 완료\\n프로그램 연결 주소:\\n\' + state[\'public_url\'], flush=True)\n            print(\'이 셀이 ▶로 돌아와도 서버는 실행 중입니다. 코랩 런타임은 유지해주세요.\')\n            return\n        if process.poll() is not None:\n            raise RuntimeError(\'공유 연결을 만들지 못했습니다. 3번 셀에서 직접 생성할 수 있습니다.\\n\' + output[-1000:])\n        time.sleep(1)\n    process.terminate()\n    raise RuntimeError(\'연결 주소를 받지 못했습니다. 4번 셀을 다시 실행하거나 3번 셀에서 직접 생성해주세요.\')\n\n\ndef split_text(text, maximum=240):\n    parts = []\n    while len(text) > maximum:\n        candidates = [m.end() for m in re.finditer(r\'[.!?。！？]\\s+|\\n+\', text[:maximum + 1]) if m.end() >= 40]\n        cut = candidates[-1] if candidates else text.rfind(\' \', 40, maximum + 1)\n        if cut < 40:\n            cut = maximum\n        parts.append(text[:cut].strip())\n        text = text[cut:].strip()\n    if text:\n        parts.append(text)\n    return parts\n\n\ndef join_wavs(paths, output, pause_ms=0):\n    parameters = None\n    with wave.open(str(output), \'wb\') as writer:\n        for number, path in enumerate(paths):\n            with wave.open(str(path), \'rb\') as reader:\n                current = (reader.getnchannels(), reader.getsampwidth(), reader.getframerate())\n                if parameters is None:\n                    parameters = current\n                    writer.setnchannels(current[0])\n                    writer.setsampwidth(current[1])\n                    writer.setframerate(current[2])\n                elif current != parameters:\n                    raise RuntimeError(\'음성 형식이 달라 원음 그대로 합칠 수 없습니다.\')\n                if number and pause_ms:\n                    writer.writeframes(b\'\\0\' * (current[2] * pause_ms // 1000) * current[0] * current[1])\n                while True:\n                    data = reader.readframes(65536)\n                    if not data:\n                        break\n                    writer.writeframes(data)\n\n\ndef serve():\n    import gc\n    import numpy as np\n    import soundfile as sf\n    import torch\n    from fastapi import FastAPI, HTTPException, Request\n    from fastapi.responses import FileResponse\n    import uvicorn\n    from qwen_voicebank_model import VoiceBankModel\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Qwen은 GPU 런타임을 사용합니다. CPU로 자동 전환하지 않았습니다.\')\n    # T4 lacks native BF16. Keep full precision there instead of forcing FP16.\n    dtype = (torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8\n             and torch.cuda.is_bf16_supported() else torch.float32)\n    model = VoiceBankModel(ROOT, dtype)\n    print(\'✅ 기본 목소리 서버 준비 완료. 생성 요청 때 선택한 목소리만 준비합니다.\', flush=True)\n    state = json.loads(STATE.read_text())\n    audio_dir = ROOT / \'audio\'\n    audio_dir.mkdir(exist_ok=True, mode=0o700)\n    jobs, lock, pending = {}, threading.RLock(), queue.Queue()\n    capacity = {\'batch_size\': 2}\n    app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)\n    prefix = \'/v1/\' + state[\'token\']\n\n    def describe(job):\n        return {key: job[key] for key in (\'status\', \'chunks_done\', \'chunks\', \'synthesis_seconds\', \'batch_size\', \'error_code\', \'phase\')}\n\n    def find_job(job_id):\n        job = jobs.get(job_id)\n        if job is None:\n            raise HTTPException(404, \'작업 없음\')\n        return job\n\n    @app.get(prefix + \'/health\')\n    def health():\n        return dict(service=SERVICE, version=VERSION, protocol=1, model=MODEL_ID, ready=True,\n                    gpu=torch.cuda.get_device_name(0), precision=str(dtype), batch_size=capacity[\'batch_size\'],\n                    bank_revision=BANK_REVISION, voice_count=len(VOICES))\n\n    @app.post(prefix + \'/prepare/{job_id}\')\n    async def prepare(job_id: str, request: Request):\n        if not re.fullmatch(\'[a-f0-9]{32}\', job_id):\n            raise HTTPException(400, \'작업 번호 오류\')\n        body = bytearray()\n        async for chunk in request.stream():\n            body.extend(chunk)\n            if len(body) > 4096:\n                raise HTTPException(413, \'요청이 너무 큼\')\n        try:\n            data = json.loads(body)\n        except (ValueError, UnicodeError):\n            raise HTTPException(400, \'JSON 형식 오류\')\n        voices = data.get(\'voices\') if isinstance(data, dict) else None\n        if (not isinstance(voices, list) or not 1 <= len(voices) <= 20\n                or any(not isinstance(voice, str) or voice not in VOICES for voice in voices)):\n            raise HTTPException(400, \'기본 목소리 선택 오류\')\n        voices = sorted(set(voices))\n        digest = hashlib.sha256(json.dumps(voices).encode()).hexdigest()\n        with lock:\n            if job_id in jobs:\n                job = jobs[job_id]\n                if job.get(\'kind\') != \'prepare\' or job[\'digest\'] != digest:\n                    raise HTTPException(409, \'작업 번호 중복\')\n                if job[\'status\'] not in (\'error\', \'cancelled\'):\n                    return describe(job)\n            if sum(job[\'status\'] in (\'queued\', \'generating\') for job in jobs.values()) >= 64 or len(jobs) >= 10000:\n                raise HTTPException(429, \'작업 대기열이 가득 참\')\n            job = dict(id=job_id, digest=digest, kind=\'prepare\', voices=voices, status=\'queued\',\n                       chunks=len(voices), chunks_done=0, synthesis_seconds=0.0, batch_size=1,\n                       error_code=\'\', phase=\'기본 목소리 준비 대기\', cancel=False)\n            jobs[job_id] = job\n            pending.put_nowait(job)\n            return describe(job)\n\n    @app.post(prefix + \'/jobs/{job_id}\')\n    async def submit(job_id: str, request: Request):\n        if not re.fullmatch(\'[a-f0-9]{32}\', job_id):\n            raise HTTPException(400, \'작업 번호 오류\')\n        body = bytearray()\n        async for chunk in request.stream():\n            body.extend(chunk)\n            if len(body) > 100000:\n                raise HTTPException(413, \'요청이 너무 큼\')\n        try:\n            data = json.loads(body)\n        except (ValueError, UnicodeError):\n            raise HTTPException(400, \'JSON 형식 오류\')\n        if not isinstance(data, dict):\n            raise HTTPException(400, \'요청 형식 오류\')\n        text, voice, instruct = data.get(\'text\'), data.get(\'voice\'), data.get(\'instruct\', \'\')\n        if (not isinstance(text, str) or not 1 <= len(text.strip()) <= 12000\n                or voice not in VOICES or instruct != \'\'):\n            raise HTTPException(400, \'대사·목소리·스타일 오류\')\n        if not model.has_reference(voice):\n            raise HTTPException(409, \'먼저 선택한 기본 목소리를 준비해주세요.\')\n        digest = hashlib.sha256(json.dumps([text, voice, instruct], ensure_ascii=False).encode()).hexdigest()\n        with lock:\n            if job_id in jobs:\n                job = jobs[job_id]\n                if job.get(\'kind\') != \'speech\' or digest != job[\'digest\']:\n                    raise HTTPException(409, \'작업 번호 중복\')\n                if job[\'status\'] not in (\'error\', \'cancelled\'):\n                    return describe(job)\n                del jobs[job_id]  # An explicit resubmission resumes a failed line.\n            if sum(job[\'status\'] in (\'queued\', \'generating\') for job in jobs.values()) >= 64 or len(jobs) >= 10000:\n                raise HTTPException(429, \'코랩 작업 대기열이 가득 참\')\n            texts = split_text(text.strip())\n            job = dict(id=job_id, digest=digest, kind=\'speech\', texts=texts, voice=voice, instruct=instruct,\n                       status=\'queued\', chunks=len(texts), chunks_done=0, synthesis_seconds=0.0,\n                       batch_size=capacity[\'batch_size\'], error_code=\'\', phase=\'대사 생성\', paths=[])\n            jobs[job_id] = job\n            pending.put_nowait(job)\n            return describe(job)\n\n    @app.get(prefix + \'/jobs/{job_id}\')\n    def job_status(job_id: str):\n        with lock:\n            return describe(find_job(job_id))\n\n    @app.post(prefix + \'/jobs/{job_id}/cancel\')\n    def cancel_job(job_id: str):\n        with lock:\n            job = find_job(job_id)\n            # Finish and save an already-started line; cancel only untouched work.\n            if job.get(\'kind\') == \'prepare\' and job[\'status\'] in (\'queued\', \'generating\'):\n                job[\'cancel\'] = True\n            if job[\'status\'] == \'queued\' and not job[\'chunks_done\']:\n                job[\'status\'] = \'cancelled\'\n            return describe(job)\n\n    @app.get(prefix + \'/jobs/{job_id}/audio\')\n    def audio(job_id: str):\n        with lock:\n            job = find_job(job_id)\n            if job[\'status\'] != \'done\' or job.get(\'kind\') != \'speech\':\n                raise HTTPException(409, \'아직 생성 중\')\n        return FileResponse(str(audio_dir / (job_id + \'.wav\')), media_type=\'audio/wav\')\n\n    def generate_batch(batch):\n        began = time.monotonic()\n        # Only this worker enters the model. Native batching avoids unsafe model\n        # calls from concurrent HTTP threads and repeated model loading.\n        with torch.inference_mode():\n            wavs, rate = model.generate_custom_voice(\n                text=[job[\'texts\'][job[\'chunks_done\']] for job in batch],\n                language=[\'Korean\'] * len(batch), speaker=[job[\'voice\'] for job in batch],\n                instruct=[job[\'instruct\'] for job in batch], non_streaming_mode=True)\n        if len(wavs) != len(batch):\n            raise RuntimeError(\'batch_count\')\n        elapsed = time.monotonic() - began\n        for job, wav in zip(batch, wavs):\n            if not len(wav) or not np.isfinite(wav).all():\n                with lock:\n                    job.update(status=\'error\', error_code=\'invalid_audio\')\n                continue\n            part = audio_dir / f"{job[\'id\']}_{job[\'chunks_done\']}.wav"\n            sf.write(str(part), wav, rate, subtype=\'PCM_16\')\n            with lock:\n                job[\'paths\'].append(part)\n                job[\'chunks_done\'] += 1\n                job[\'synthesis_seconds\'] += elapsed\n                job[\'batch_size\'] = len(batch)\n                if job[\'chunks_done\'] == job[\'chunks\']:\n                    output = audio_dir / (job[\'id\'] + \'.wav\')\n                    join_wavs(job[\'paths\'], output)\n                    job[\'status\'] = \'done\'\n                    for path in job[\'paths\']:\n                        path.unlink(missing_ok=True)\n                    job[\'paths\'] = []\n                    job[\'texts\'] = []\n                else:\n                    pending.put_nowait(job)\n\n    def process(batch):\n        memory_failure = False\n        try:\n            generate_batch(batch)\n        except torch.cuda.OutOfMemoryError:\n            memory_failure = True\n        except Exception:\n            import traceback\n            traceback.print_exc()\n            with lock:\n                for job in batch:\n                    if job[\'status\'] != \'done\':\n                        job.update(status=\'error\', error_code=\'generation\')\n        # Leave the exception handler before retrying, releasing its traceback\n        # and tensors so the smaller batch can actually reclaim GPU memory.\n        if memory_failure:\n            gc.collect()\n            torch.cuda.empty_cache()\n            if len(batch) > 1:\n                capacity[\'batch_size\'] = 1\n                print(\'GPU 메모리 부족: 모델·정밀도는 유지하고 배치 크기만 1개로 줄입니다.\', flush=True)\n                for job in batch:\n                    process([job])\n            else:\n                with lock:\n                    batch[0].update(status=\'error\', error_code=\'memory\')\n\n    def worker():\n        while True:\n            first = pending.get()\n            if first.get(\'kind\') == \'prepare\':\n                from concurrent.futures import CancelledError\n                with lock:\n                    if first[\'status\'] in (\'cancelled\', \'error\', \'done\'):\n                        continue\n                    first[\'status\'] = \'generating\'\n                def progress(phase, done, total):\n                    with lock:\n                        first.update(phase=phase, chunks_done=done, chunks=total)\n                try:\n                    model.prepare(first[\'voices\'], progress, lambda: first[\'cancel\'])\n                    with lock:\n                        first[\'status\'] = \'done\'\n                except CancelledError:\n                    with lock:\n                        first[\'status\'] = \'cancelled\'\n                except Exception:\n                    import traceback\n                    traceback.print_exc()\n                    with lock:\n                        first.update(status=\'error\', error_code=\'preparation\')\n                gc.collect()\n                torch.cuda.empty_cache()\n                continue\n            batch = [first]\n            deadline = time.monotonic() + 0.12\n            while len(batch) < capacity[\'batch_size\']:\n                try:\n                    candidate = pending.get(timeout=max(0.001, deadline - time.monotonic()))\n                    if candidate.get(\'kind\') == \'prepare\':\n                        pending.put_nowait(candidate)\n                        break\n                    batch.append(candidate)\n                except queue.Empty:\n                    break\n                if time.monotonic() >= deadline:\n                    break\n            with lock:\n                batch = [job for job in batch if job[\'status\'] not in (\'cancelled\', \'done\', \'error\')]\n                for job in batch:\n                    job[\'status\'] = \'generating\'\n            if batch:\n                process(batch)\n\n    threading.Thread(target=worker, name=\'qwen-native-batch\', daemon=True).start()\n    uvicorn.run(app, host=\'127.0.0.1\', port=state[\'port\'], access_log=False, log_level=\'warning\')\n\n\ndef direct(plan_path):\n    """Generate inside a Colab cell, without any external web connection."""\n    plan = json.loads(Path(plan_path).read_text(encoding=\'utf-8-sig\'))\n    if (plan.get(\'format\') != \'voice-studio-qwen-plan-v1\' or plan.get(\'model\') != MODEL_ID\n            or plan.get(\'bank_revision\') != BANK_REVISION):\n        raise ValueError(\'공유 사이트에서 받은 Qwen 코랩 직접 생성용 JSON 파일을 올려주세요.\')\n    items = plan.get(\'items\', [])\n    if not items or len(items) > 5000:\n        raise ValueError(\'대사는 1~5,000개까지 넣어주세요.\')\n    indices = [item[\'index\'] for item in items]\n    if any(type(index) is not int or index <= 0 for index in indices) or len(set(indices)) != len(indices):\n        raise ValueError(\'대사 번호가 올바르지 않습니다.\')\n    items.sort(key=lambda item: item[\'index\'])\n    voices = sorted(set(item[\'voice\'] for item in items))\n    prepare_id = hashlib.sha256((\'prepare:\' + BANK_REVISION + json.dumps(voices)).encode()).hexdigest()[:32]\n    status = local_request(\'/prepare/\' + prepare_id, {\'voices\': voices})\n    started, last_message = time.monotonic(), \'\'\n    while status[\'status\'] != \'done\':\n        if status[\'status\'] in (\'error\', \'cancelled\'):\n            raise RuntimeError(\'기본 목소리 준비가 중단되었습니다. 5번 오류 로그를 확인해주세요.\')\n        message = f"{status.get(\'phase\', \'준비 중\')} · {status.get(\'chunks_done\', 0)}/{len(voices)}"\n        if message != last_message:\n            print(message, flush=True)\n            last_message = message\n        if time.monotonic() - started > 7200:\n            raise RuntimeError(\'목소리 준비 응답이 2시간을 넘었습니다. 5번 오류 로그를 확인해주세요.\')\n        time.sleep(1)\n        status = local_request(\'/jobs/\' + prepare_id)\n    pause_ms = max(0, min(3000, int(plan.get(\'pause_ms\', 500))))\n    folder = ROOT / \'direct\' / hashlib.sha256(Path(plan_path).read_bytes()).hexdigest()[:16]\n    folder.mkdir(parents=True, exist_ok=True)\n    waiting, active, finished = deque(items), {}, {}\n    while waiting or active:\n        while waiting and len(active) < 4:\n            item = waiting.popleft()\n            output = folder / f"{item[\'index\']:04d}.wav"\n            if output.is_file():\n                finished[item[\'index\']] = output\n                continue\n            job_id = hashlib.sha256((str(folder) + str(item[\'index\'])).encode()).hexdigest()[:32]\n            payload = {name: item[name] for name in (\'text\', \'voice\', \'instruct\')}\n            local_request(\'/jobs/\' + job_id, payload)\n            active[job_id] = (item, output, time.monotonic())\n        for job_id, (item, output, submitted_at) in list(active.items()):\n            status = local_request(\'/jobs/\' + job_id)\n            if status[\'status\'] in (\'error\', \'cancelled\'):\n                raise RuntimeError(f"{item[\'index\']}번 생성 중단: {status.get(\'error_code\')}. 완료 음성은 {folder}에 보관했습니다.")\n            if time.monotonic() - submitted_at > 1800:\n                raise RuntimeError(f"{item[\'index\']}번이 30분 동안 완료되지 않았습니다. 5번 셀에서 로그를 확인해주세요. 완료 음성은 보관합니다.")\n            if status[\'status\'] == \'done\':\n                import shutil\n                temporary = output.with_suffix(\'.part.wav\')\n                shutil.copyfile(ROOT / \'audio\' / (job_id + \'.wav\'), temporary)\n                temporary.replace(output)\n                finished[item[\'index\']] = output\n                del active[job_id]\n                print(f"✅ {len(finished)}/{len(items)}개 저장 완료 · {item[\'index\']}번 {item.get(\'speaker\', \'\')}", flush=True)\n        if active:\n            time.sleep(1)\n    merged = folder / \'전체_음성.wav\'\n    join_wavs([finished[item[\'index\']] for item in items], merged, pause_ms)\n    mp3 = folder / \'전체_음성.mp3\'\n    subprocess.check_call([\'ffmpeg\', \'-nostdin\', \'-v\', \'error\', \'-y\', \'-i\', str(merged),\n                           \'-c:a\', \'libmp3lame\', \'-b:a\', \'192k\', str(mp3)])\n    def timestamp(seconds):\n        ms = round(seconds * 1000)\n        return f\'{ms // 3600000:02d}:{ms // 60000 % 60:02d}:{ms // 1000 % 60:02d},{ms % 1000:03d}\'\n    entries, cursor = [], 0.0\n    for number, item in enumerate(items, 1):\n        with wave.open(str(finished[item[\'index\']]), \'rb\') as reader:\n            end = cursor + reader.getnframes() / reader.getframerate()\n        text = (item.get(\'speaker\', \'\') + \': \' if plan.get(\'include_speaker\') else \'\') + item[\'text\']\n        entries.append(f\'{number}\\n{timestamp(cursor)} --> {timestamp(end)}\\n{text}\\n\')\n        cursor = end + pause_ms / 1000\n    (folder / \'자막.srt\').write_text(\'\\n\'.join(entries), encoding=\'utf-8-sig\')\n    import zipfile\n    bundle = folder / \'Qwen_전체음성_자막.zip\'\n    with zipfile.ZipFile(bundle, \'w\', zipfile.ZIP_STORED) as archive:\n        for path in (mp3, folder / \'자막.srt\'):\n            archive.write(path, path.name)\n    save_json(ROOT / \'direct_result.json\', {\'mp3\': str(mp3), \'wav\': str(merged), \'bundle\': str(bundle)})\n    print(\'✅ 전체 대사를 순서대로 합쳤습니다: \' + str(mp3), flush=True)\n\n\ndef backup():\n    import zipfile\n    bank = ROOT / \'voices\' / BANK_REVISION\n    archive_path = ROOT / \'Qwen_기본목소리20종_보관.zip\'\n    count = 0\n    with zipfile.ZipFile(archive_path, \'w\', zipfile.ZIP_STORED) as archive:\n        archive.writestr(\'manifest.json\', json.dumps({\'format\': \'voice-studio-qwen-bank-v1\', \'revision\': BANK_REVISION}))\n        for voice in VOICES:\n            audio = bank / (voice + \'.wav\')\n            metadata = audio.with_suffix(\'.json\')\n            if audio.is_file() and metadata.is_file():\n                archive.write(audio, audio.name)\n                archive.write(metadata, metadata.name)\n                count += 1\n    print(f\'✅ 준비한 목소리 {count}개를 보관했습니다: {archive_path}\', flush=True)\n\n\ndef restore(archive_path):\n    import io\n    import zipfile\n    from qwen_voicebank_catalog import REFERENCE_TEXT\n    if STATE.is_file():\n        state = json.loads(STATE.read_text())\n        pid = state.get(\'pid\')\n        command = Path(f\'/proc/{pid}/cmdline\') if isinstance(pid, int) else None\n        if command and command.is_file() and b\'qwen_voicebank_colab_server.py\\x00serve\' in command.read_bytes():\n            raise RuntimeError(\'복원은 새 런타임에서 1번 설치 → 6번 복원 → 2번 서버 준비 순서로 진행해주세요.\')\n    with zipfile.ZipFile(archive_path) as archive:\n        infos = archive.infolist()\n        if len(infos) > 41 or sum(info.file_size for info in infos) > 160 * 1024 * 1024:\n            raise ValueError(\'목소리 보관 파일 크기가 올바르지 않습니다.\')\n        if any(info.file_size > 8 * 1024 * 1024 for info in infos):\n            raise ValueError(\'목소리 보관 항목이 너무 큽니다.\')\n        manifest = json.loads(archive.read(\'manifest.json\'))\n        if manifest.get(\'format\') != \'voice-studio-qwen-bank-v1\' or manifest.get(\'revision\') != BANK_REVISION:\n            raise ValueError(\'현재 기본 목소리 목록의 보관 파일이 아닙니다.\')\n        names = set(archive.namelist())\n        allowed = {\'manifest.json\'} | {voice + suffix for voice in VOICES for suffix in (\'.wav\', \'.json\')}\n        if not names <= allowed or len(names) != len(infos):\n            raise ValueError(\'보관 파일 안의 항목이 올바르지 않습니다.\')\n        accepted = []\n        for voice in VOICES:\n            if voice + \'.wav\' not in names:\n                continue\n            raw = archive.read(voice + \'.wav\')\n            meta = archive.read(voice + \'.json\')\n            identity = hashlib.sha256(json.dumps([BANK_REVISION, REFERENCE_TEXT, VOICEBANK[voice]],\n                                sort_keys=True, ensure_ascii=False).encode()).hexdigest()\n            if json.loads(meta).get(\'identity\') != identity:\n                raise ValueError(\'목소리 설명과 보관 음성이 맞지 않습니다.\')\n            with wave.open(io.BytesIO(raw), \'rb\') as reader:\n                frames, rate = reader.getnframes(), reader.getframerate()\n                if rate <= 0 or not 2 <= frames / rate <= 30:\n                    raise ValueError(\'보관 음성 길이가 올바르지 않습니다.\')\n                reader.setpos(frames - 1)\n                if len(reader.readframes(1)) != reader.getsampwidth() * reader.getnchannels():\n                    raise ValueError(\'보관 음성 파일이 잘렸습니다.\')\n            accepted.append((voice, raw, meta))\n    bank = ROOT / \'voices\' / BANK_REVISION\n    bank.mkdir(parents=True, exist_ok=True, mode=0o700)\n    for voice, raw, meta in accepted:\n        (bank / (voice + \'.wav\')).write_bytes(raw)\n        (bank / (voice + \'.json\')).write_bytes(meta)\n    print(f\'✅ 목소리 {len(accepted)}개를 복원했습니다. 2번 셀부터 이어서 실행하세요.\', flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'action\', choices=[\'setup\', \'start\', \'serve\', \'tunnel\', \'direct\', \'backup\', \'restore\'])\n    parser.add_argument(\'--plan\')\n    args = parser.parse_args()\n    if args.action == \'direct\':\n        direct(args.plan)\n    elif args.action == \'restore\':\n        restore(args.plan)\n    else:\n        globals()[args.action]()\n',
    'qwen_voicebank_model.py': '"""One-GPU VoiceDesign -> Base lifecycle, with durable reference audio reuse."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nfrom qwen_voicebank_catalog import BASE_MODEL, BANK_REVISION, DESIGN_MODEL, REFERENCE_TEXT, VOICEBANK\n\n\nclass VoiceBankModel:\n    def __init__(self, root, dtype):\n        self.root = Path(root) / \'voices\' / BANK_REVISION\n        self.root.mkdir(parents=True, exist_ok=True, mode=0o700)\n        self.dtype = dtype\n        self.model = None\n        self.model_id = None\n        self.prompts = {}\n\n    def reference_path(self, voice):\n        return self.root / (voice + \'.wav\')\n\n    def _identity(self, voice):\n        return hashlib.sha256(json.dumps([BANK_REVISION, REFERENCE_TEXT, VOICEBANK[voice]],\n                                        sort_keys=True, ensure_ascii=False).encode()).hexdigest()\n\n    def has_reference(self, voice):\n        if voice not in VOICEBANK:\n            return False\n        try:\n            import soundfile as sf\n            path = self.reference_path(voice)\n            metadata = json.loads(path.with_suffix(\'.json\').read_text())\n            info = sf.info(str(path))\n            return metadata.get(\'identity\') == self._identity(voice) and info.frames > 0 and 2 <= info.duration <= 30\n        except (OSError, ValueError, RuntimeError):\n            return False\n\n    def _load(self, model_id):\n        if self.model_id == model_id and self.model is not None:\n            return\n        import gc\n        import torch\n        from qwen_tts import Qwen3TTSModel\n        # Never retain VoiceDesign and Base on the same GPU at the same time.\n        self.prompts.clear()\n        self.model = None\n        self.model_id = None\n        gc.collect()\n        torch.cuda.empty_cache()\n        print(\'▶ 모델 준비: \' + model_id, flush=True)\n        self.model = Qwen3TTSModel.from_pretrained(model_id, device_map=\'cuda:0\',\n                         dtype=self.dtype, attn_implementation=\'sdpa\')\n        self.model.model.eval()\n        self.model_id = model_id\n\n    def prepare(self, voices, progress, cancelled=lambda: False):\n        import numpy as np\n        import soundfile as sf\n        import torch\n        from concurrent.futures import CancelledError\n        voices = list(dict.fromkeys(voices))\n        missing = [voice for voice in voices if not self.has_reference(voice)]\n        if missing:\n            progress(\'새 목소리 모델 다운로드·준비\', len(voices) - len(missing), len(voices))\n            self._load(DESIGN_MODEL)\n            for position, voice in enumerate(missing):\n                if cancelled():\n                    raise CancelledError()\n                progress(\'첫 목소리 준비 · \' + VOICEBANK[voice][\'name\'], len(voices) - len(missing) + position, len(voices))\n                # A fixed seed helps reproduce the profile on a new runtime.\n                # The saved WAV, rather than a fresh design, anchors later lines.\n                with torch.random.fork_rng(devices=[0]), torch.inference_mode():\n                    torch.manual_seed(VOICEBANK[voice][\'seed\'])\n                    wavs, rate = self.model.generate_voice_design(text=REFERENCE_TEXT, language=\'Korean\',\n                        instruct=VOICEBANK[voice][\'instruct\'], non_streaming_mode=True)\n                if len(wavs) != 1 or not np.isfinite(wavs[0]).all() or not 2 <= len(wavs[0]) / rate <= 30:\n                    raise RuntimeError(\'참조 음성이 정상 길이로 생성되지 않았습니다. 자동으로 반복 생성하지 않습니다.\')\n                target = self.reference_path(voice)\n                temporary = target.with_suffix(\'.part.wav\')\n                sf.write(str(temporary), wavs[0], rate, subtype=\'PCM_16\')\n                temporary.replace(target)\n                metadata = target.with_suffix(\'.json\')\n                temp_meta = metadata.with_suffix(\'.tmp\')\n                temp_meta.write_text(json.dumps({\'identity\': self._identity(voice)}, ensure_ascii=False))\n                temp_meta.replace(metadata)\n                del wavs\n        if cancelled():\n            raise CancelledError()\n        progress(\'대사 생성용 Base 모델 준비\', len(voices), len(voices))\n        self._load(BASE_MODEL)\n        for voice in voices:\n            if cancelled():\n                raise CancelledError()\n            self._prompt(voice)\n        progress(\'선택한 목소리 준비 완료\', len(voices), len(voices))\n\n    def _prompt(self, voice):\n        if voice not in self.prompts:\n            if not self.has_reference(voice):\n                raise RuntimeError(\'먼저 선택한 기본 목소리 준비를 완료해주세요.\')\n            self.prompts[voice] = self.model.create_voice_clone_prompt(\n                ref_audio=str(self.reference_path(voice)), ref_text=REFERENCE_TEXT,\n                x_vector_only_mode=False)[0]\n        return self.prompts[voice]\n\n    def generate_custom_voice(self, *, text, language, speaker, instruct, non_streaming_mode):\n        self._load(BASE_MODEL)\n        prompts = [self._prompt(voice) for voice in speaker]\n        # Base does not accept CustomVoice style instructions. Keep the designed\n        # reference\'s tone; never pretend a style selector controls this model.\n        return self.model.generate_voice_clone(text=text, language=language,\n                    voice_clone_prompt=prompts, non_streaming_mode=non_streaming_mode)\n',
    'qwen_voicebank_catalog.py': '"""Selectable Korean voice designs; references are prepared once, not per line."""\nBANK_REVISION = "korean-cast-20261001-v1"\nREFERENCE_TEXT = "오늘은 마을에 전해 내려오는 이야기를 들려드리겠습니다. 사람들은 저마다의 사연을 품고 살아가고 있었습니다."\nDESIGN_MODEL = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"\nBASE_MODEL = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"\n\n# These are designed voice profiles, not extra pretrained CustomVoice speaker IDs.\n# A single cached reference per profile anchors the voice across all dialogue.\n_ROWS = [\n    ("M01", "남성", "도윤 · 차분한 청년", "20대 · 부드러운 중저음", "A Korean man in his late twenties, a smooth warm low tenor, gentle breath support, measured clear delivery."),\n    ("M02", "남성", "지후 · 밝은 청년", "20대 · 밝고 가벼운 중음", "A Korean man in his early twenties, a bright light tenor with an open forward resonance, lively but unhurried diction."),\n    ("M03", "남성", "태준 · 단정한 선비", "30대 · 맑고 단정한 음색", "A Korean man in his thirties, a clean focused midrange tenor, refined scholarly composure, precise consonants and relaxed vowels."),\n    ("M04", "남성", "건우 · 듬직한 장정", "30대 · 두텁고 힘 있는 저음", "A Korean man in his thirties, a thick resonant bass-baritone, robust chest resonance, steady understated strength without shouting."),\n    ("M05", "남성", "성민 · 따뜻한 아버지", "40대 · 포근하고 깊은 중저음", "A Korean man in his forties, a mellow rounded baritone with velvety resonance, reassuring paternal warmth and easy clear diction."),\n    ("M06", "남성", "진석 · 엄정한 사또", "50대 · 묵직하고 또렷한 저음", "A Korean man in his fifties, a deep sonorous bass, firm deliberate articulation, dignified authority with a calm controlled volume."),\n    ("M07", "남성", "만수 · 구수한 장터 상인", "40대 · 거칠고 구수한 중음", "A Korean man in his forties, a textured lightly raspy midrange baritone, friendly conversational storytelling, rounded vowels, no exaggerated accent."),\n    ("M08", "남성", "덕수 · 온화한 어르신", "60대 · 부드러운 노년 음색", "A Korean man in his late sixties, a gentle slightly weathered low baritone, soft grain and reassuring warmth, steady supported speech without shaking."),\n    ("M09", "남성", "영호 · 노련한 이야기꾼", "70대 · 담백한 노년 중음", "A Korean man in his seventies, a light aged midrange voice with a dry airy edge, articulate seasoned storytelling, stable rhythm without mumbling."),\n    ("M10", "남성", "시우 · 소년", "10대 후반 · 맑고 높은 음색", "A Korean teenage boy about seventeen, a youthful high tenor, clear light resonance, sincere natural delivery, no exaggerated cartoon performance."),\n    ("F01", "여성", "서연 · 고운 여성 해설", "30대 · 고운 아나운서 중음", "A Korean woman in her thirties, an elegant smooth mezzo-soprano, clear broadcast diction, warm composed storytelling suitable for older listeners."),\n    ("F02", "여성", "하린 · 밝은 아가씨", "20대 · 밝고 맑은 고음", "A Korean woman in her early twenties, a bright crystalline soprano with light forward resonance, cheerful clean articulation without squeakiness."),\n    ("F03", "여성", "소담 · 수줍은 청년", "20대 · 가늘고 부드러운 중고음", "A Korean woman in her twenties, a delicate soft high mezzo voice, intimate gentle resonance and restrained emotion, clear diction without whispering."),\n    ("F04", "여성", "연화 · 단아한 여인", "30대 · 맑고 단단한 중음", "A Korean woman in her thirties, a pure centered mezzo voice with a firm clean tone, graceful poised articulation and even unhurried cadence."),\n    ("F05", "여성", "미정 · 따뜻한 어머니", "40대 · 포근하고 둥근 중저음", "A Korean woman in her forties, a warm rounded low mezzo, comforting maternal resonance with soft consonants and clearly shaped words."),\n    ("F06", "여성", "정희 · 위엄 있는 안주인", "50대 · 단단하고 깊은 저음", "A Korean woman in her fifties, a rich strong contralto, composed dignified authority, precise decisive consonants without harshness."),\n    ("F07", "여성", "복순 · 정겨운 이웃", "50대 · 구수하고 탄력 있는 중음", "A Korean woman in her fifties, a textured slightly husky middle register, friendly earthy resonance, natural conversational rhythm and clear speech."),\n    ("F08", "여성", "순덕 · 다정한 할머니", "60대 · 포근한 노년 중저음", "A Korean woman in her late sixties, a mellow aged low mezzo with a soft velvety grain, tender storytelling, stable support without trembling."),\n    ("F09", "여성", "옥분 · 옛이야기 할머니", "70대 · 담백한 노년 중고음", "A Korean woman in her seventies, a light weathered middle-high voice with a dry delicate grain, expressive traditional storytelling, clear steady words."),\n    ("F10", "여성", "나래 · 소녀", "10대 후반 · 가볍고 맑은 음색", "A Korean teenage girl about seventeen, a youthful light soprano, soft clear resonance, earnest natural delivery, no exaggerated cartoon performance."),\n]\nVOICEBANK = {key: dict(gender=gender, name=name, description=description,\n                      seed=37100 + index, instruct=direction +\n                      " Speak native Korean with standard Korean pronunciation. Read only the supplied text. "\n                      "Use natural clear storytelling, with no music, sound effects or background voices.")\n             for index, (key, gender, name, description, direction) in enumerate(_ROWS)}\n',
    'qwen_colab_output.py': '"""Forward runner output to a Colab cell and show verified completion details.\n\nOnly standard-library imports; importing this file never starts a process.\nThe notebooks embed this helper so downloaded copies remain self-contained.\n"""\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nimport urllib.request\n\n\ndef _ready_state(runner):\n    expected_service = {\n        \'qwen_colab_server.py\': \'voice-studio-qwen-customvoice\',\n        \'qwen_voicebank_colab_server.py\': \'voice-studio-qwen-voicebank\',\n    }[runner.name]\n    try:\n        state = json.loads((runner.parent / \'state.json\').read_text(encoding=\'utf-8\'))\n        port, token = state[\'port\'], state[\'token\']\n        if not isinstance(port, int) or not 1 <= port <= 65535:\n            raise ValueError(\'Invalid local port\')\n        if not isinstance(token, str) or not re.fullmatch(r\'[A-Za-z0-9_-]{20,100}\', token):\n            raise ValueError(\'Invalid local token\')\n        request = urllib.request.Request(f\'http://127.0.0.1:{port}/v1/{token}/health\')\n        with urllib.request.urlopen(request, timeout=15) as response:\n            status = json.load(response)\n        if status.get(\'service\') != expected_service or not status.get(\'ready\'):\n            raise ValueError(\'Server is not ready\')\n        return state, status\n    except (OSError, ValueError, KeyError, TypeError):\n        raise RuntimeError(\'서버 준비 상태를 확인하지 못했습니다. 2번 셀을 실행하고 위에 표시되는 오류를 확인해주세요.\') from None\n\n\ndef run_qwen(runner, action, *arguments):\n    runner = Path(runner)\n    labels = {\n        \'setup\': \'1번 설치\', \'start\': \'2번 서버 준비\',\n        \'direct\': \'3번 전체 음성 생성\', \'tunnel\': \'4번 연결 주소 만들기\',\n        \'backup\': \'목소리 보관\', \'restore\': \'목소리 복원\',\n    }\n    if action not in labels:\n        raise ValueError(\'지원하지 않는 코랩 작업입니다.\')\n    if not runner.is_file():\n        raise RuntimeError(\'먼저 수정본의 1번 설치 셀을 실행해주세요.\')\n    print(f\'▶ {labels[action]} 시작 · 진행 내용이 아래에 표시됩니다.\', flush=True)\n    command = [sys.executable, \'-u\', str(runner), action, *map(str, arguments)]\n    # Notebook kernels do not always display inherited subprocess file descriptors.\n    # Read the child\'s output explicitly and print through the notebook\'s stream.\n    process = subprocess.Popen(\n        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n        text=True, encoding=\'utf-8\', errors=\'replace\', bufsize=1,\n        env=dict(os.environ, PYTHONUNBUFFERED=\'1\'),\n    )\n    try:\n        for line in process.stdout:\n            print(line, end=\'\', flush=True)\n        returncode = process.wait()\n    except KeyboardInterrupt:\n        process.terminate()\n        try:\n            process.wait(timeout=5)\n        except subprocess.TimeoutExpired:\n            process.kill()\n            process.wait()\n        print(\'\\n셀 대기를 중단했습니다. 다시 실행하면 서버 상태를 먼저 확인합니다.\', flush=True)\n        raise\n    finally:\n        process.stdout.close()\n    if returncode != 0:\n        raise RuntimeError(f\'{labels[action]} 중 오류가 발생했습니다. 바로 위 오류 내용을 확인해주세요. (종료 코드 {returncode})\')\n    # A zero exit code alone is not proof that the requested resource is ready.\n    if action == \'setup\':\n        if not (runner.parent / \'installed.json\').is_file() or not (runner.parent / \'venv/bin/python\').is_file():\n            raise RuntimeError(\'설치 완료 파일이 없습니다. 위 설치 로그를 확인하고 1번 셀을 다시 실행해주세요.\')\n        print(\'✅ 1번 설치 완료. 이제 2번 셀의 ▶를 눌러주세요.\', flush=True)\n    elif action == \'start\':\n        _, status = _ready_state(runner)\n        print(f"✅ 2번 서버 응답 확인 완료 · {status.get(\'gpu\', \'\')}", flush=True)\n        print(\'공유 사이트에서 사용하려면 3번을 건너뛰고 4번 셀의 ▶를 눌러주세요.\', flush=True)\n    elif action == \'tunnel\':\n        state, _ = _ready_state(runner)\n        url = state.get(\'public_url\', \'\')\n        if not isinstance(url, str) or not re.fullmatch(r\'https://[a-z0-9-]+\\.trycloudflare\\.com/v1/[A-Za-z0-9_-]{20,100}\', url) or not url.endswith(\'/v1/\' + state[\'token\']):\n            raise RuntimeError(\'연결 주소가 저장되지 않았습니다. 위 로그를 확인하고 4번 셀을 다시 실행해주세요.\')\n        label = (\'기본 목소리 20종 · 내 코랩 주소\'\n                 if runner.name == \'qwen_voicebank_colab_server.py\' else \'Qwen3-TTS · 내 코랩 주소\')\n        print(\'\\n✅ 프로그램 연결 주소 — 아래 한 줄 전체를 복사하세요.\', flush=True)\n        print(url, flush=True)\n        print(f\'공유 사이트 왼쪽 「{label}」 칸에 붙여넣고 연결 확인을 누르세요.\', flush=True)\n    # Return None so the notebook does not replace useful output with CompletedProcess.\n',
}
for qwen_name, qwen_source in qwen_sources.items():
    qwen_file = qwen_root / qwen_name
    qwen_file.write_text(qwen_source, encoding="utf-8")
    qwen_file.chmod(0o600)
qwen_runner = qwen_root / 'qwen_voicebank_colab_server.py'
run_qwen = run_path(str(qwen_root / "qwen_colab_output.py"))["run_qwen"]
run_qwen(qwen_runner, "setup")


In [ ]:
#@title 2️⃣ 기본 목소리 서버 준비 (목소리는 첫 생성 때 준비)
from pathlib import Path
from runpy import run_path
qwen_runner = Path('/content/voice_studio_qwen_bank_v1/qwen_voicebank_colab_server.py')
if not qwen_runner.is_file():
    raise RuntimeError('먼저 1번 설치 셀을 실행해주세요.')
qwen_output = qwen_runner.with_name('qwen_colab_output.py')
if not qwen_output.is_file():
    raise RuntimeError('수정본의 1번 설치 셀을 먼저 실행해주세요.')
run_qwen = run_path(str(qwen_output))['run_qwen']
run_qwen(qwen_runner, 'start')


## 3번은 코랩 안에서 직접 생성할 때 사용합니다
공유 사이트에서 사용할 경우 **3번을 건너뛰고 4번**으로 이동하세요.

직접 생성은 사이트에서 내려받은 `Qwen_기본목소리20종_대본.json`을 올리면 됩니다. 완료된 대사는 같은 런타임에서 재사용하고, 전부 완료되면 순서대로 MP3 하나로 합칩니다. 런타임이 삭제되면 저장한 결과도 사라지므로 다운로드하세요.


In [ ]:
#@title 3️⃣ 코랩에서 대본 전체 생성 → MP3 하나로 다운로드 (선택)
from google.colab import files
from IPython.display import Audio, display, FileLink
from pathlib import Path
from runpy import run_path
import json
qwen_root = Path('/content/voice_studio_qwen_bank_v1')
qwen_runner = qwen_root / 'qwen_voicebank_colab_server.py'
if not qwen_runner.is_file():
    raise RuntimeError('1번과 2번 셀을 먼저 실행해주세요.')
qwen_output = qwen_runner.with_name('qwen_colab_output.py')
if not qwen_output.is_file():
    raise RuntimeError('수정본의 1번 설치 셀을 먼저 실행해주세요.')
run_qwen = run_path(str(qwen_output))['run_qwen']
print('공유 사이트에서 받은 기본 목소리 20종 JSON 대본 파일 1개를 올려주세요.')
qwen_uploaded = files.upload()
qwen_json_files = [name for name in qwen_uploaded if name.lower().endswith('.json')]
if len(qwen_json_files) != 1:
    raise ValueError('Qwen JSON 파일 1개만 선택해주세요.')
qwen_plan_path = qwen_root / 'direct_input.json'
qwen_plan_path.write_bytes(qwen_uploaded[qwen_json_files[0]])
run_qwen(qwen_runner, 'direct', '--plan', qwen_plan_path)
qwen_result = json.loads((qwen_root / 'direct_result.json').read_text())
print('✅ 전체 MP3 생성 완료. 다운로드를 시작합니다.')
display(Audio(filename=qwen_result['mp3']))
files.download(qwen_result['mp3'])
print('MP3와 자막을 함께 보관하려면 아래 ZIP 파일도 받을 수 있습니다.')
display(FileLink(qwen_result['bundle']))


## 4번은 공유 사이트에서 생성할 때 사용합니다
무료 코랩은 외부 웹 화면을 주로 사용하는 작업이 종료될 수 있습니다. 무료로 코랩 안에서 직접 사용하려면 위 3번을 이용하세요.

아래에 나오는 **프로그램 연결 주소 전체**를 공유 사이트 왼쪽 **기본 목소리 20종 · 내 코랩 주소**에 넣고 **기본 목소리 20종 연결 확인**을 누르세요. 다른 이용자에게 주소를 보내지 말고, 각자 본인 코랩을 실행하세요.


In [ ]:
#@title 4️⃣ 공유 사이트 연결 주소 만들기 (선택)
from pathlib import Path
from runpy import run_path
qwen_runner = Path('/content/voice_studio_qwen_bank_v1/qwen_voicebank_colab_server.py')
if not qwen_runner.is_file():
    raise RuntimeError('먼저 1번과 2번 셀을 실행해주세요.')
qwen_output = qwen_runner.with_name('qwen_colab_output.py')
if not qwen_output.is_file():
    raise RuntimeError('수정본의 1번 설치 셀을 먼저 실행해주세요.')
run_qwen = run_path(str(qwen_output))['run_qwen']
run_qwen(qwen_runner, 'tunnel')


In [ ]:
#@title 5️⃣ 오류 로그 보기 (오류가 났을 때만)
from pathlib import Path
qwen_log = Path('/content/voice_studio_qwen_bank_v1/server.log')
print(qwen_log.read_text(errors='replace')[-7000:] if qwen_log.is_file() else '아직 모델 준비 로그가 없습니다. 1번과 2번 셀부터 확인해주세요.')


In [ ]:
#@title 6️⃣ 준비한 목소리 보관 / 새 런타임에 복원 (선택)
작업 = "보관 파일 받기" #@param ["보관 파일 받기", "보관 파일 복원"]
from google.colab import files
from pathlib import Path
from runpy import run_path
qwen_root = Path('/content/voice_studio_qwen_bank_v1')
qwen_runner = qwen_root / 'qwen_voicebank_colab_server.py'
if not qwen_runner.is_file():
    raise RuntimeError('먼저 1번 설치 셀을 실행해주세요.')
qwen_output = qwen_runner.with_name('qwen_colab_output.py')
if not qwen_output.is_file():
    raise RuntimeError('수정본의 1번 설치 셀을 먼저 실행해주세요.')
run_qwen = run_path(str(qwen_output))['run_qwen']
if 작업 == '보관 파일 받기':
    run_qwen(qwen_runner, 'backup')
    files.download(str(qwen_root / 'Qwen_기본목소리20종_보관.zip'))
else:
    print('새 런타임에서 1번 설치 후, 2번 서버 준비 전에 보관 ZIP 파일을 복원하세요.')
    qwen_backup_upload = files.upload()
    if len(qwen_backup_upload) != 1:
        raise ValueError('목소리 보관 ZIP 파일 1개만 선택해주세요.')
    qwen_backup_data = next(iter(qwen_backup_upload.values()))
    if len(qwen_backup_data) > 100 * 1024 * 1024:
        raise ValueError('보관 파일은 100MB 이하여야 합니다.')
    qwen_backup_path = qwen_root / 'restore_input.zip'
    qwen_backup_path.write_bytes(qwen_backup_data)
    run_qwen(qwen_runner, 'restore', '--plan', qwen_backup_path)
